In [1]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from src.local_config import *
from src.trading_signal import *

In [2]:
# Load in data from previous notebook
df1_is = pd.read_csv(PROJECT_ROOT / "data/df1_is",
                     index_col=0, parse_dates=True)
df1_oos = pd.read_csv(PROJECT_ROOT / "data/df1_oos",
                      index_col=0, parse_dates=True)

df2_is = pd.read_csv(PROJECT_ROOT / "data/df2_is",
                     index_col=0, parse_dates=True)
df2_oos = pd.read_csv(PROJECT_ROOT / "data/df2_oos",
                      index_col=0, parse_dates=True)

static_results_df_is = pd.read_csv(PROJECT_ROOT / "data/static_hedge_ratio_is")
dynamic_results_df_is = pd.read_csv(PROJECT_ROOT / "data/dynamic_hedge_ratio_is")

static_results_df_oos = pd.read_csv(PROJECT_ROOT / "data/static_hedge_ratio_oos")
dynamic_results_df_oos = pd.read_csv(PROJECT_ROOT / "data/dynamic_hedge_ratio_oos")


In [3]:
# Load dictionaries from previous notebook
# In-Sample
with open(PROJECT_ROOT / "data/dictionaries/dynamic_spreads_is.pkl", "rb") as f:
    dynamic_spreads_is = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/dynamic_details_is.pkl", "rb") as f:
    dynamic_details_is = pickle.load(f)

# Out-Of-Sample
with open(PROJECT_ROOT / "data/dictionaries/dynamic_spreads_oos.pkl", "rb") as f:
    dynamic_spreads_oos = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/dynamic_details_oos.pkl", "rb") as f:
    dynamic_details_oos = pickle.load(f)

# Load Portfolio Weighting
portfolio    = pd.read_csv(PROJECT_ROOT / "data/portfolio")
portfolio_pairs = set(portfolio["pair"])

# Implementation - Trading Signal
- For backtest here and compare the Kalman indicator to the baseline and a simple long strategy of only Asset A and Asset B (backtest.py)
- We implement a basic trading signal using standardised values for our dynamic spreads (z-scores)

The general rule is that:
- If the z-score > +2 - we short the spread 
- If the z-score < -2 - we long the spread
- If the z-score = 0 - we exit our position 

Mathematically, we can express it in a mapping:
$$
\text{Position}(z) =
\begin{cases}
-1 & \text{if } z > 2 \\
+1 & \text{if } z < -2 \\
\text{hold previous position} & \text{otherwise}
\end{cases}
$$

In [4]:
entry_values = [1.5, 2.0, 2.5]
exit_values = [0.0, 0.5, 1.0]

is_results = {}
is_summary_rows = []

for e in entry_values:
    for x in exit_values:
        key = f"entry_{e}_exit_{x}"

        is_signals_dict = generate_kalman_signals(
            dynamic_details_is,
            entry_z=e,
            exit_z=x
        )

        is_results[key] = is_signals_dict

        for (pair_name, R), df in is_signals_dict.items():
            is_summary_rows.append({
                "strategy": key,
                "pair": pair_name,
                "obs_cov": R,
                "entry_z": e,
                "exit_z": x,
                "num_trades": df["position"].diff().abs().sum() / 2,
                "mean_zscore": df["zscore"].mean(),
                "std_zscore": df["zscore"].std()
            })

is_trading_signals = pd.DataFrame(is_summary_rows)

In [5]:
is_summary_rows

[{'strategy': 'entry_1.5_exit_0.0',
  'pair': '1347 HK Equity vs 268 HK Equity',
  'obs_cov': 0.5,
  'entry_z': 1.5,
  'exit_z': 0.0,
  'num_trades': np.float64(34.5),
  'mean_zscore': np.float64(0.04327970837414214),
  'std_zscore': np.float64(1.2462208042615563)},
 {'strategy': 'entry_1.5_exit_0.0',
  'pair': '1347 HK Equity vs 268 HK Equity',
  'obs_cov': 1.0,
  'entry_z': 1.5,
  'exit_z': 0.0,
  'num_trades': np.float64(33.5),
  'mean_zscore': np.float64(0.037982670529391055),
  'std_zscore': np.float64(1.2675657454897022)},
 {'strategy': 'entry_1.5_exit_0.0',
  'pair': '1347 HK Equity vs 268 HK Equity',
  'obs_cov': 5.0,
  'entry_z': 1.5,
  'exit_z': 0.0,
  'num_trades': np.float64(35.5),
  'mean_zscore': np.float64(0.023074719237687183),
  'std_zscore': np.float64(1.2953814766983487)},
 {'strategy': 'entry_1.5_exit_0.0',
  'pair': '857 HK Equity vs 2386 HK Equity',
  'obs_cov': 0.5,
  'entry_z': 1.5,
  'exit_z': 0.0,
  'num_trades': np.float64(33.5),
  'mean_zscore': np.float64(0

In [6]:
is_trading_signals

,strategy,pair,obs_cov,entry_z,exit_z,num_trades,mean_zscore,std_zscore
0,entry_1.5_exit_0.0,1347 HK Equity vs 268 HK Equity,0.5,1.5,0.0,34.5,0.043280,1.246221
1,entry_1.5_exit_0.0,1347 HK Equity vs 268 HK Equity,1.0,1.5,0.0,33.5,0.037983,1.267566
2,entry_1.5_exit_0.0,1347 HK Equity vs 268 HK Equity,5.0,1.5,0.0,35.5,0.023075,1.295381
3,entry_1.5_exit_0.0,857 HK Equity vs 2386 HK Equity,0.5,1.5,0.0,33.5,0.028883,1.255986
4,entry_1.5_exit_0.0,857 HK Equity vs 2386 HK Equity,1.0,1.5,0.0,27.5,0.030947,1.268819
...,...,...,...,...,...,...,...,...
103,entry_2.5_exit_1.0,3993 HK Equity vs 2689 HK Equity,1.0,2.5,1.0,19.5,0.034924,1.281620
104,entry_2.5_exit_1.0,3993 HK Equity vs 2689 HK Equity,5.0,2.5,1.0,20.5,0.023837,1.306160
105,entry_2.5_exit_1.0,1258 HK Equity vs 3899 HK Equity,0.5,2.5,1.0,24.0,-0.003021,1.285520
106,entry_2.5_exit_1.0,1258 HK Equity vs 3899 HK Equity,1.0,2.5,1.0,23.0,-0.018303,1.311623


In [7]:
# Save trading_signal details 
with open(PROJECT_ROOT / "data/dictionaries/is_trading_signals.pkl", "wb") as f:
    pickle.dump(is_trading_signals, f) 

with open(PROJECT_ROOT / "data/dictionaries/is_results.pkl", "wb") as f:
    pickle.dump(is_results, f)

# OOS Signals

In [8]:
# Prepend last 60 IS rows to each OOS pair so rolling(60) has warmup data,
# avoiding the ~59-day gap at the start of OOS signals
dynamic_details_portfolio_is = {
    k: v for k, v in dynamic_details_is.items()
    if k[0] in portfolio_pairs
}

dynamic_details_portfolio_oos = {
    k: v for k, v in dynamic_details_oos.items()
    if k[0] in portfolio_pairs
}

print("IS  pairs:", sorted(set(k[0] for k in dynamic_details_portfolio_is)))
print("OOS pairs:", sorted(set(k[0] for k in dynamic_details_portfolio_oos)))

IS  pairs: ['1258 HK Equity vs 3899 HK Equity', '1347 HK Equity vs 268 HK Equity', '3993 HK Equity vs 2689 HK Equity', '857 HK Equity vs 2386 HK Equity']
OOS pairs: ['1258 HK Equity vs 3899 HK Equity', '1347 HK Equity vs 268 HK Equity', '3993 HK Equity vs 2689 HK Equity', '857 HK Equity vs 2386 HK Equity']


In [9]:
entry_values = [1.5, 2.0, 2.5]
exit_values  = [0.0, 0.5, 1.0]

oos_results      = {}
oos_summary_rows = []

for e in entry_values:
    for x in exit_values:
        key = f"entry_{e}_exit_{x}"

        oos_signals_dict = generate_kalman_signals(
            dynamic_details_portfolio_oos,
            entry_z=e,
            exit_z=x
        )

        oos_results[key] = oos_signals_dict

        for (pair_name, R), df in oos_signals_dict.items():
            oos_summary_rows.append({
                "strategy":  key,
                "pair":      pair_name,
                "obs_cov":   R,
                "entry_z":   e,
                "exit_z":    x,
                "num_trades": df["position"].diff().abs().sum() / 2,
                "mean_zscore": df["zscore"].mean(),
                "std_zscore":  df["zscore"].std()
            })

oos_trading_signals = pd.DataFrame(oos_summary_rows)

In [10]:
oos_trading_signals

,strategy,pair,obs_cov,entry_z,exit_z,num_trades,mean_zscore,std_zscore
0,entry_1.5_exit_0.0,1347 HK Equity vs 268 HK Equity,0.5,1.5,0.0,12.5,0.059432,1.271094
1,entry_1.5_exit_0.0,1347 HK Equity vs 268 HK Equity,1.0,1.5,0.0,12.5,0.056150,1.303117
2,entry_1.5_exit_0.0,1347 HK Equity vs 268 HK Equity,5.0,1.5,0.0,12.5,0.033886,1.339131
3,entry_1.5_exit_0.0,857 HK Equity vs 2386 HK Equity,0.5,1.5,0.0,11.5,0.043486,1.381552
4,entry_1.5_exit_0.0,857 HK Equity vs 2386 HK Equity,1.0,1.5,0.0,9.5,0.079229,1.399114
...,...,...,...,...,...,...,...,...
103,entry_2.5_exit_1.0,3993 HK Equity vs 2689 HK Equity,1.0,2.5,1.0,6.0,0.066604,1.269258
104,entry_2.5_exit_1.0,3993 HK Equity vs 2689 HK Equity,5.0,2.5,1.0,7.0,0.134893,1.305270
105,entry_2.5_exit_1.0,1258 HK Equity vs 3899 HK Equity,0.5,2.5,1.0,12.5,0.067926,1.265890
106,entry_2.5_exit_1.0,1258 HK Equity vs 3899 HK Equity,1.0,2.5,1.0,11.5,0.100038,1.293254


In [11]:
oos_results

{'entry_1.5_exit_0.0': {('1347 HK Equity vs 268 HK Equity',
   0.5):                    y         x   alpha_t    beta_t  spread_t  obs_cov  \
  Date                                                                    
  2022-12-28  3.305641  2.829678  0.492194  1.025860 -0.089408      0.5   
  2022-12-29  3.291010  2.813011  0.498853  1.021991 -0.082714      0.5   
  2022-12-30  3.296503  2.817801  0.505005  1.018398 -0.078146      0.5   
  2023-01-03  3.311090  2.897016  0.520426  1.010331 -0.136280      0.5   
  2023-01-04  3.305641  2.937043  0.542024  0.999365 -0.171563      0.5   
  ...              ...       ...       ...       ...       ...      ...   
  2025-12-23  4.252772  2.599722  1.736627  0.951586  0.042285      0.5   
  2025-12-24  4.268998  2.591516  1.737604  0.952190  0.063779      0.5   
  2025-12-29  4.282897  2.579459  1.739035  0.952952  0.085760      0.5   
  2025-12-30  4.320816  2.588516  1.740688  0.954016  0.110642      0.5   
  2025-12-31  4.308111  2.587012 

In [12]:
# Save trading_signal details 
with open(PROJECT_ROOT / "data/dictionaries/oos_trading_signals.pkl", "wb") as f:
    pickle.dump(oos_trading_signals, f) 

with open(PROJECT_ROOT / "data/dictionaries/oos_results.pkl", "wb") as f:
    pickle.dump(oos_results, f)